# Российские и китайские LLM через Caila: базовое сравнение

Локальный Jupyter или Colab, обычный CPU. Модели исполняются через API Caila.
Шесть кандидатов: GigaChat-2 / Pro, YandexGPT Lite / Pro, Qwen3 Instruct, DeepSeek V3.1 Chat.
Проверяем micro-F1, precision/recall, exact match, JSON и задержку.

**Только baseline:** один `bench.PROMPT`, тот же системный текст, что у локальных LLM.
Без few-shot, подбора промптов, обучения и генерации объявлений.
Каждая пара «модель × test» автоматически сохраняется в ClearML.
Выполняйте ячейки сверху вниз. Ключи вводятся в скрытых полях.

## 1. Настройки

Повторное измерение уже проверенной Qwen3-235B-A22B Instruct на новом test v7
из dev@de0e000: 80 объявлений, 20 с пустым ответом. test_v7 совпадает с test_gold.
Используется тот же системный промпт, temperature=0, max_tokens=96 и контракт
Pydantic. Полный прогон всех80 строк, без отсева по первым30 и без переноса
ответов с другой версии текста. Train не используется. Результаты сохраняются
в отдельной группе qwen235-test-v7 и ClearML, старые результаты сохраняются.


In [ ]:
PARTICIPANT = "member-api"
RUN_GROUP = "qwen235-hard-v0"
PROJECT = "avito-microcategories"
MODEL_NAMES = ["qwen3-235b-instruct"]
DIRTY_TEST = None
CLEAN_TEST = "data/hard/hard_v0.xlsx"
ASSUME_SOURCE = 101
LIMIT = None
AUTO_UPLOAD = True
INCLUDE_DATASET = True
BASE_URL = "https://caila.io/api/adapters/openai"
USE_DRIVE_IN_COLAB = True
EARLY_STOP_MIN_ROWS = 0
TARGET_F1 = 0.93
SCREEN_F1 = 0.93

REUSE_RUN_GROUP = None  # None для полностью нового платного прогона


## 2. Рабочая папка

Локально запустите Jupyter из клона репозитория. В Colab при первом запуске
загрузите `caila-baseline-bundle.zip`, приложенный к ноутбуку: в нём код и публичный грязный test.
Так Colab получает именно эту версию стенда, включая ещё не опубликованные изменения.
На Drive сохраняются результаты, окружение остаётся на временном диске Colab.

In [ ]:
from pathlib import Path
import os, sys, json, re, subprocess, importlib.util, zipfile

os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUNBUFFERED"] = "1"
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content/caila-baseline")
    ROOT.mkdir(exist_ok=True)
    if not (ROOT / "research/experiments/api_bench.py").is_file():
        from google.colab import files
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Загрузите один caila-baseline-bundle.zip")
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                target = (ROOT / member.filename).resolve()
                if ROOT.resolve() not in target.parents and target != ROOT.resolve():
                    raise ValueError("Недопустимый путь в архиве")
            archive.extractall(ROOT)
    if USE_DRIVE_IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = Path("/content/drive/MyDrive/caila-baselines")
    else:
        WORK = ROOT / "research/api-runs"
else:
    roots = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in roots if (p / "research/experiments/api_bench.py").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Запустите Jupyter из корня репозитория Innoglobalhack")
    WORK = ROOT / "research/api-runs"

CODE = ROOT / "research/experiments"
if not (CODE / "api_bench.py").is_file():
    raise RuntimeError("В комплекте отсутствует api_bench.py")
for value in (PARTICIPANT, RUN_GROUP):
    if not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError("Используйте латинские буквы, цифры, _ или - в имени участника/группы")
RUN_ROOT = WORK / PARTICIPANT / RUN_GROUP
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Результаты:", RUN_ROOT)

## 3. Окружение CPU-клиента

Изолированный Python с NumPy, sklearn, Parquet и ClearML. CUDA, Transformers
и веса моделей не устанавливаются. Уже существующее окружение используется повторно.

In [ ]:
ENV_DIR = Path("/content/caila-api-env") if IN_COLAB else WORK / ".venv-api"
API_PYTHON = str(ENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python"))
if not Path(API_PYTHON).is_file():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(ENV_DIR)], check=True)
subprocess.run([API_PYTHON, "-m", "pip", "install", "--disable-pip-version-check",
                "-r", str(CODE / "requirements-api.txt")], check=True)
subprocess.run([API_PYTHON, "-c", "import numpy, sklearn, pyarrow, clearml; print('CPU-клиент готов')"], check=True)

## 4. Конфигурация моделей и проверка данных без запросов

Исходный текст не обрезается. Точное описание предобработки находится в manifest.
Сравнивайте модели на одной версии test; переход dirty → clean меняет сам тест.
Train здесь не используется. Файл с незавершённой разметкой не передавайте как clean.

In [ ]:
all_models = json.loads((CODE / "api_models.json").read_text(encoding="utf-8"))
by_name = {entry["name"]: entry for entry in all_models}
if not MODEL_NAMES or len(MODEL_NAMES) != len(set(MODEL_NAMES)) or set(MODEL_NAMES) - by_name.keys():
    raise ValueError("MODEL_NAMES содержит неизвестные имена, повторы или пуст")
SELECTED_MODELS = [by_name[name] for name in MODEL_NAMES]
CONFIG_DIR = RUN_ROOT / "configs"
CONFIG_DIR.mkdir(exist_ok=True)
ALL_CONFIG = CONFIG_DIR / "selected.json"
ALL_CONFIG.write_text(json.dumps(SELECTED_MODELS, ensure_ascii=False, indent=2), encoding="utf-8")
for entry in SELECTED_MODELS:
    (CONFIG_DIR / (entry["name"] + ".json")).write_text(
        json.dumps([entry], ensure_ascii=False, indent=2), encoding="utf-8")

def data_path(value):
    p = Path(value)
    p = p if p.is_absolute() else ROOT / p
    if p.suffix.lower() == ".xlsx":
        exported = RUN_ROOT / "input" / (p.stem + ".jsonl")
        subprocess.run([API_PYTHON, str(CODE / "export_hard_test.py"),
                        "--input", str(p), "--out", str(exported)], check=True)
        return exported
    return p

DATA_ARGS = ["--assume-source", str(ASSUME_SOURCE)]
if DIRTY_TEST is None:
    DATA_ARGS += ["--skip-dirty"]
else:
    DATA_ARGS += ["--dirty", str(data_path(DIRTY_TEST))]
if CLEAN_TEST is not None:
    DATA_ARGS += ["--clean", str(data_path(CLEAN_TEST))]
if LIMIT is not None:
    DATA_ARGS += ["--limit", str(LIMIT)]
COMMON_ARGS = ["--base-url", BASE_URL, "--clearml-project", PROJECT,
               "--participant", PARTICIPANT, "--run-group", RUN_GROUP,
               "--early-stop-min-rows", str(EARLY_STOP_MIN_ROWS),
               "--target-f1", str(TARGET_F1), "--screen-f1", str(SCREEN_F1)]
if not AUTO_UPLOAD:
    COMMON_ARGS += ["--no-upload"]
if not INCLUDE_DATASET:
    COMMON_ARGS += ["--no-dataset-upload"]
subprocess.run([API_PYTHON, str(CODE / "api_bench.py"), "--config", str(ALL_CONFIG),
                *DATA_ARGS, "--dry-run"], check=True)
print("Модели:", ", ".join(MODEL_NAMES))
subprocess.run([API_PYTHON, "-c", "import bench; print(bench.PROMPT)"], cwd=CODE, check=True)

## 5. Ключи и read-only проверка доступа

Ключ Caila не выводится. ClearML использует существующую конфигурацию или личные
credentials из скрытых полей. Проверка не создаёт задач и не вызывает генерацию.
Отсутствие модели в каталоге ключа останавливает запуск до платных запросов.
В Colab с зарубежным IP сервис может ограничить доступ: в таком случае используйте
локальный запуск с уже настроенным доступом Caila.

In [ ]:
from getpass import getpass

if not os.environ.get("CAILA_API_KEY"):
    os.environ["CAILA_API_KEY"] = getpass("Caila API key: ").strip()
if not os.environ["CAILA_API_KEY"]:
    raise ValueError("Пустой ключ Caila")

if AUTO_UPLOAD:
    os.environ.setdefault("CLEARML_API_HOST", "https://api.clear.ml")
    os.environ.setdefault("CLEARML_WEB_HOST", "https://app.clear.ml")
    os.environ.setdefault("CLEARML_FILES_HOST", "https://files.clear.ml")
    clearml_config = Path(os.environ.get("CLEARML_CONFIG_FILE", str(Path.home() / "clearml.conf"))).expanduser()
    if not clearml_config.is_file():
        for key in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
            if not os.environ.get(key):
                os.environ[key] = getpass(key + ": ").strip()
    else:
        os.environ["CLEARML_CONFIG_FILE"] = str(clearml_config.resolve())

subprocess.run([API_PYTHON, str(CODE / "api_bench.py"), "--config", str(ALL_CONFIG),
                *DATA_ARGS, *COMMON_ARGS, "--check"], check=True)

## 6. Базовые прогоны и автоматическое сохранение в ClearML

Следующая ячейка действительно вызывает платный API. По умолчанию — все 208 строк
для каждой выбранной модели. Используется текущий общий промпт из dev; старые результаты хранятся отдельно.
Каждая законченная пара модель/test выгружается синхронно; ссылка сохраняется локально.
При перезапуске завершённые строки используются повторно. Не запускайте одну группу параллельно.

In [ ]:
def run_logged(cmd, name):
    folder = RUN_ROOT / name
    folder.mkdir(parents=True, exist_ok=True)
    log_path = folder / "process.log"
    print("Лог процесса:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\n--- invocation ---\n")
        log.flush()
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            code = process.wait()
        log.write(f"Exit code: {code}\n")
        log.flush()
    if code:
        raise subprocess.CalledProcessError(code, cmd)

def run_model(name):
    if name not in MODEL_NAMES:
        raise ValueError("Модель отсутствует в MODEL_NAMES")
    reuse_args = []
    if REUSE_RUN_GROUP is not None:
        if not re.fullmatch(r"[A-Za-z0-9_-]+", REUSE_RUN_GROUP) or REUSE_RUN_GROUP == RUN_GROUP:
            raise ValueError("REUSE_RUN_GROUP должна быть другой корректной группой")
        previous = WORK / PARTICIPANT / REUSE_RUN_GROUP / name
        if previous.is_dir():
            reuse_args = ["--reuse-from", str(previous)]
    run_logged([API_PYTHON, "-u", str(CODE / "api_bench.py"),
                    "--config", str(CONFIG_DIR / (name + ".json")),
                    "--out", str(RUN_ROOT / name), *DATA_ARGS, *COMMON_ARGS, *reuse_args], name)

def export_model(name, force=False):
    if name not in MODEL_NAMES:
        raise ValueError("Модель отсутствует в MODEL_NAMES")
    cmd = [API_PYTHON, "-u", str(CODE / "api_bench.py"), "--export-only",
           "--out", str(RUN_ROOT / name), "--clearml-project", PROJECT]
    if not INCLUDE_DATASET:
        cmd += ["--no-dataset-upload"]
    if force:
        cmd += ["--force-upload"]
    run_logged(cmd, name)

In [ ]:
failed_models = []
for name in MODEL_NAMES:
    print("\nМодель:", name, flush=True)
    try:
        run_model(name)
    except subprocess.CalledProcessError:
        failed_models.append(name)
        print("Прогон или выгрузка не завершены:", name)
run_status = {"state": "completed_with_errors" if failed_models else "completed",
              "failed_models": failed_models}
(RUN_ROOT / "run_status.json").write_text(json.dumps(run_status, ensure_ascii=False, indent=2), encoding="utf-8")
if failed_models:
    print("Часть прогонов не завершена:", ", ".join(failed_models))
    print("Причины сохранены в process.log соответствующей модели; таблица ниже содержит готовые результаты.")

## 7. Итоговая таблица и ссылки

Результаты dirty и clean показываются раздельно. `invalid_rate` нужен вместе с F1:
некорректный ответ учитывается как пустой список, а не как корректный отказ.
Если генерация завершилась, но выгрузка оборвалась, вызовите `export_model("имя")`;
это не обращается к Caila. `force=True` создаёт новую задачу, если старую удалили.
Результаты `early_stopped` относятся только к обработанному префиксу.
Они исключаются из сравнения полного test; причина сохранена в `evaluation.json`.


In [ ]:
from IPython.display import display, Markdown
import csv

leaderboard = []
for path in sorted(RUN_ROOT.glob("*/*/*/metrics.json")):
    scores = json.loads(path.read_text(encoding="utf-8"))
    row = {key: scores[key] for key in ("dataset", "model", "rows", "micro_f1", "precision", "recall",
                                       "exact_match", "invalid_rate", "latency_p95_s", "test_hash")}
    receipt = path.parent / "clearml.json"
    row["legacy_strict_micro_f1"] = scores.get("legacy_strict_micro_f1", scores["micro_f1"])
    row["legacy_strict_invalid_rate"] = scores.get("legacy_strict_invalid_rate", scores["invalid_rate"])
    row["reused_rows"] = scores.get("reused_rows", 0)
    row["status"] = scores.get("evaluation_status", "completed")
    row["planned_rows"] = scores.get("planned_rows", scores["rows"])
    row["clearml"] = json.loads(receipt.read_text(encoding="utf-8"))["url"] if receipt.is_file() else "не выгружено"
    leaderboard.append(row)
leaderboard.sort(key=lambda row: (row["dataset"], row["status"] != "completed", -row["micro_f1"]))
if not leaderboard:
    raise RuntimeError("Завершённых результатов пока нет")
with (RUN_ROOT / "leaderboard.csv").open("w", encoding="utf-8-sig", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(leaderboard[0]))
    writer.writeheader()
    writer.writerows(leaderboard)
lines = ["| Test | Модель | Статус | N / план | micro-F1 | Legacy F1 | Precision | Recall | Invalid | ClearML |",
         "|---|---|---|---:|---:|---:|---:|---:|---:|---|"]
for row in leaderboard:
    link = "[задача](" + row["clearml"] + ")" if row["clearml"].startswith("https://") else row["clearml"]
    lines.append(f"| {row['dataset']} | {row['model']} | {row['status']} | {row['rows']} / {row['planned_rows']} | {row['micro_f1']:.4f} | {row['legacy_strict_micro_f1']:.4f} | "
                 f"{row['precision']:.4f} | {row['recall']:.4f} | {row['invalid_rate']:.3f} | {link} |")
display(Markdown("\n".join(lines)))
print("early_stopped: F1 только по просмотренному префиксу; не итоговый результат полного test.")
print("Таблица:", RUN_ROOT / "leaderboard.csv")